# Chatbot WebSocket Demo & Testing

Demo WebSocket streaming API với full features: **profile_json**, **test_json**, **user_id**, **cancel**, **message queue**.

### Prerequisites
```bash
uvicorn app.main:app --reload --port 8000
```

In [1]:
import json
import asyncio
import websockets
from datetime import datetime

WS_URL = "ws://localhost:8000"
ORIGIN = "http://localhost:8000"

## 1. Định Nghĩa Profile & Holland Data

In [3]:
# Profile JSON - Form Tư Vấn Hướng Nghiệp
profile_json = {
    "form_date": "2026-01-21",
    "basic_info": {
        "full_name": "Nguyễn Minh Anh",
        "age": 18,
        "address": "Hà Nội",
        "status": "hoc_sinh"
    },
    "self_understanding": {
        "hobbies": [
            {"name": "Viết blog", "reason": "Thích chia sẻ suy nghĩ"},
            {"name": "Edit video", "reason": "Sáng tạo nội dung"},
            {"name": "Nghe podcast", "reason": "Học hỏi kiến thức"}
        ],
        "strengths": ["Viết tốt", "Giao tiếp ổn", "Nhạy cảm với cảm xúc"],
        "favorite_subjects": [
            {"name": "Tiếng Anh", "reason": "Thích giao tiếp quốc tế"},
            {"name": "Tâm lý", "reason": "Hiểu con người"},
            {"name": "Truyền thông", "reason": "Sáng tạo"}
        ],
        "experience": "CTV truyền thông CLB trường"
    },
    "career_orientation": {
        "has_chosen": False,
        "considering_careers": "Tâm lý học / Marketing",
        "target_schools": ["ĐH Khoa học Xã hội và Nhân văn", "ĐH Kinh tế Quốc dân"],
        "important_factors": ["Phù hợp tính cách", "Cơ hội việc làm", "Thu nhập"]
    },
    "family_info": {
        "father_job": "Kỹ sư",
        "mother_job": "Giáo viên",
        "family_expectations": "Gia đình muốn ngành ổn định",
        "has_guidance": True,
        "guidance_content": "Bố mẹ khuyên học kinh tế hoặc công nghệ"
    }
}

# Holland Test JSON
test_json = {
    "result_id": "result_001",
    "test_id": "holland_v1",
    "created_at": "2026-01-21T10:00:00Z",
    "scores": {"R": 14, "I": 18, "A": 26, "S": 20, "E": 9, "C": 11},
    "top3": ["A", "S", "I"],
    "top_code": "ASI"
}

print("Profile:", profile_json["basic_info"]["full_name"])
print("Holland Code:", test_json["top_code"])

Profile: Nguyễn Minh Anh
Holland Code: ASI


## 2. Helper: WebSocket Chat Function

Hàm helper xử lý đầy đủ tất cả message types:
- `start` — bắt đầu stream
- `token` — từng token từ LLM
- `tool_start` — LLM gọi tool (search, research...)
- `end` — kết thúc stream, kèm token usage
- `error` — lỗi
- `queued` — tin nhắn bị xếp hàng (LLM đang bận)
- `cancelled` — cancel thành công

In [4]:
async def ws_chat(
    thread_id: str,
    message: str,
    user_id: str = None,
    profile: dict = None,
    holland: dict = None,
    timeout: float = 120.0,
) -> dict:
    """
    Send a message via WebSocket and collect the full streamed response.
    
    Returns:
        dict with keys: response, tokens, usage, events
    """
    uri = f"{WS_URL}/ws/chat/{thread_id}"
    payload = {"message": message}
    if user_id:
        payload["user_id"] = user_id
    if profile:
        payload["profile_json"] = profile
    if holland:
        payload["test_json"] = holland
    
    result = {"response": "", "tokens": 0, "usage": {}, "events": []}
    
    print(f"Connecting to {uri}...")
    try:
        async with websockets.connect(uri, origin=ORIGIN) as ws:
            await ws.send(json.dumps(payload))
            print(f"Sent: {message[:60]}...\n")
            
            async with asyncio.timeout(timeout):
                async for raw in ws:
                    data = json.loads(raw)
                    t = data.get("type", "unknown")
                    result["events"].append(t)
                    
                    if t == "start":
                        print("[Stream Started]", end=" ", flush=True)
                    
                    elif t == "token":
                        content = data.get("content", "")
                        print(content, end="", flush=True)
                        result["response"] += content
                        result["tokens"] += 1
                    
                    elif t == "tool_start":
                        tool = data.get("tool", "unknown")
                        print(f"\n  🔧 [Tool: {tool}]", end=" ", flush=True)
                    
                    elif t == "end":
                        result["usage"] = data.get("usage", {})
                        print("\n\n[Stream Ended]")
                        break
                    
                    elif t == "error":
                        err = data.get("content", data.get("error", "unknown"))
                        print(f"\n❌ Error: {err}")
                        break
                    
                    elif t == "queued":
                        pos = data.get("position", "?")
                        print(f"\n⏳ Queued (position {pos}): {data.get('message', '')}")
                    
                    elif t == "cancelled":
                        print("\n🛑 Cancelled")
                        break
                    
                    else:
                        print(f"\n  [{t}] {list(data.keys())}")
    
    except TimeoutError:
        print(f"\n⏰ Timeout after {timeout}s!")
    except Exception as e:
        print(f"\n💥 WebSocket Error: {e}")
    
    return result

## 3. Chat cơ bản (với profile + Holland)

In [5]:
result = await ws_chat(
    thread_id="demo-ws-001",
    message="Chào cô, em đang phân vân giữa Tâm lý học và Marketing. Em nên chọn hướng nào?",
    profile=profile_json,
    holland=test_json,
)

print(f"\n--- Stats ---")
print(f"Tokens: {result['tokens']}, Response: {len(result['response'])} chars")
if result['usage']:
    u = result['usage']
    print(f"Prompt: {u.get('prompt_tokens')}, Completion: {u.get('completion_tokens')}, Cost: ${u.get('total_cost_usd')}")

Connecting to ws://localhost:8000/ws/chat/demo-ws-001...

💥 WebSocket Error: Multiple exceptions: [Errno 10061] Connect call failed ('::1', 8000, 0, 0), [Errno 10061] Connect call failed ('127.0.0.1', 8000)

--- Stats ---
Tokens: 0, Response: 0 chars


## 4. Chat với user_id (Long-term Memory)

Gửi `user_id` để kích hoạt cross-session memory. LLM sẽ nhớ user qua các session khác nhau.

In [ ]:
result = await ws_chat(
    thread_id="demo-ws-memory",
    message="Em muốn biết điểm chuẩn ngành Marketing năm 2025 ở ĐH Kinh tế Quốc dân",
    user_id="user_minh_anh",
    profile=profile_json,
    holland=test_json,
)

print(f"\n--- Stats ---")
print(f"Tokens: {result['tokens']}, Response: {len(result['response'])} chars")

## 5. Follow-up (cùng thread_id)

Gửi tin nhắn thứ 2 trên cùng `thread_id` — LLM sẽ nhớ ngữ cảnh trước đó.

In [ ]:
result = await ws_chat(
    thread_id="demo-ws-memory",
    message="Vậy học phí ra sao? So sánh giữa 2 trường em đang quan tâm.",
    user_id="user_minh_anh",
    profile=profile_json,
    holland=test_json,
)

print(f"\n--- Stats ---")
print(f"Tokens: {result['tokens']}, Response: {len(result['response'])} chars")

## 6. Chat không có profile (chỉ message)

Tất cả fields ngoài `message` đều optional.

In [ ]:
result = await ws_chat(
    thread_id="demo-ws-bare",
    message="Ngành AI có triển vọng không?",
)

print(f"\n--- Stats ---")
print(f"Tokens: {result['tokens']}, Response: {len(result['response'])} chars")

## 7. Cancel mid-stream

Gửi `{"type": "cancel"}` trong khi LLM đang generate để dừng.

In [ ]:
async def demo_cancel():
    thread_id = f"demo-cancel-{int(datetime.now().timestamp())}"
    uri = f"{WS_URL}/ws/chat/{thread_id}"
    
    payload = {
        "message": "Hãy viết một bài luận thật dài về triết học phương Đông.",
        "profile_json": profile_json,
        "test_json": test_json,
    }
    
    print(f"Connecting to {uri}...")
    async with websockets.connect(uri, origin=ORIGIN) as ws:
        await ws.send(json.dumps(payload))
        print("Sent message. Will cancel after 5 tokens...\n")
        
        tokens = 0
        async with asyncio.timeout(60):
            async for raw in ws:
                data = json.loads(raw)
                t = data.get("type")
                
                if t == "start":
                    print("[Started]", end=" ", flush=True)
                elif t == "token":
                    print(data.get("content", ""), end="", flush=True)
                    tokens += 1
                    if tokens >= 5:
                        print(f"\n\n>>> Sending CANCEL after {tokens} tokens...")
                        await ws.send(json.dumps({"type": "cancel"}))
                elif t == "cancelled":
                    print("✅ Cancel confirmed!")
                    break
                elif t == "end":
                    print("\n[Ended before cancel — LLM was too fast]")
                    break
                elif t == "error":
                    print(f"\n❌ Error: {data.get('content', '')}")
                    break
    
    print(f"\nTokens received before cancel: {tokens}")

await demo_cancel()

## 8. Message Queue (gửi tin nhắn mới khi LLM đang bận)

Gửi tin nhắn thứ 2 trong khi LLM đang xử lý tin nhắn 1.  
Server sẽ trả `{"type": "queued"}` và xử lý tin nhắn 2 sau khi tin nhắn 1 xong.

In [ ]:
async def demo_message_queue():
    thread_id = f"demo-queue-{int(datetime.now().timestamp())}"
    uri = f"{WS_URL}/ws/chat/{thread_id}"
    
    payload1 = {
        "message": "Giải thích lý thuyết tương đối của Einstein.",
        "profile_json": profile_json,
        "test_json": test_json,
    }
    payload2 = {
        "message": "Cảm ơn! Bây giờ hãy nói về ngành Vật lý.",
        "profile_json": profile_json,
        "test_json": test_json,
    }
    
    print(f"Connecting to {uri}...")
    async with websockets.connect(uri, origin=ORIGIN) as ws:
        # Send message 1
        await ws.send(json.dumps(payload1))
        print("Sent msg 1. Will send msg 2 after 5 tokens...\n")
        
        tokens = 0
        phase = 1
        queued_ok = False
        
        async with asyncio.timeout(240):
            async for raw in ws:
                data = json.loads(raw)
                t = data.get("type")
                
                if t == "start" and phase == 1:
                    print("[Msg 1 Started]", end=" ", flush=True)
                elif t == "token" and phase == 1:
                    print(data.get("content", ""), end="", flush=True)
                    tokens += 1
                    if tokens == 5:
                        print(f"\n\n>>> Sending msg 2 while LLM is busy...")
                        await ws.send(json.dumps(payload2))
                elif t == "queued":
                    print(f"⏳ QUEUED: {data.get('message', '')}")
                    queued_ok = True
                elif t == "end" and phase == 1:
                    print(f"\n\n[Msg 1 Ended] Waiting for msg 2 from queue...\n")
                    phase = 2
                elif t == "start" and phase == 2:
                    print("[Msg 2 Started]", end=" ", flush=True)
                elif t == "token" and phase == 2:
                    print(data.get("content", ""), end="", flush=True)
                elif t == "end" and phase == 2:
                    print(f"\n\n[Msg 2 Ended]")
                    break
                elif t == "error":
                    print(f"\n❌ Error: {data.get('content', data.get('error', ''))}")
                    break
    
    print(f"\nQueued event received: {'YES ✅' if queued_ok else 'NO ❌'}")

await demo_message_queue()

## API Reference

### WebSocket Endpoint
```
ws://host/ws/chat/{thread_id}
```

### Client → Server Message
```json
{
    "message": "string (required)",
    "user_id": "string (optional, enables cross-session memory)",
    "profile_json": {"basic_info": {...}, ...},
    "test_json": {"scores": {...}, "top_code": "ASI", ...},
    "test_id": "string (optional, resolve test_json from DB)",
    "history": [{...}]
}
```

### Cancel Request
```json
{"type": "cancel"}
```

### Server → Client Events
| Type | Description |
|-|-|
| `start` | Stream bắt đầu |
| `token` | Từng token từ LLM (`content` field) |
| `tool_start` | LLM gọi tool (`tool`, `input` fields) |
| `end` | Stream kết thúc (kèm `usage` object) |
| `error` | Lỗi (`content` hoặc `error` field) |
| `queued` | Tin nhắn bị xếp hàng (`position`, `message`) |
| `cancelled` | Cancel thành công |